Lab 3
=====

*Introduction to Data Science and Visualization*


### Objectives

1.  Manipulate datasets
2.  Wrangle datasets

### Tool

Python, in particular the `pandas` library:

In [1]:
import pandas as pd
import openpyxl ## it is not used in the code but it is required to read xlsx files with pandas

### Dataset

We use the public dataset about Italian employment data from ISTAT with updated to January 2026, published on Marc 4, 2026:

<https://www.istat.it/comunicato-stampa/occupati-e-disoccupati-dati-provvisori-gennaio-2026>


Download the file corresponding to ["Serie Storiche"](https://www.istat.it/wp-content/uploads/2026/03/202601_serie-storiche.xlsx)

In [2]:
import os
import urllib.request

url = 'https://www.istat.it/wp-content/uploads/2026/03/202601_serie-storiche.xlsx'
file = url.split("/")[-1]
if not os.path.exists(file):
    urllib.request.urlretrieve(url, file)

## Reading

Read from the first sheet (`Tab1`) the main indicators (seasonally adjusted).
Pay attention to the initial rows that contain descriptions and not data.

Suggestions:
- specify the name (or the number of the sheet)
- pass as the third parameter
  - the portion of the dataset you want to analyze, indicating the cell coordinates at the ends of the main diagonal, e.g. `"B2:E10"`
  - the number of initial rows to skip

In [12]:
df_file = pd.read_excel(file,
                   sheet_name = "Tab1",
                   usecols= "A:P",
                   skiprows= 5)

The columns are grouped and represent respectively *All* (columns 1-6), *Males* (columns 7-11) and *Females* (columns 12-16).

In each group, the columns represent:

- Reference year
- Reference month
- Activity rate
- Employment rate
- General unemployment rate
- Youth unemployment rate (15-24 years) (only for *All*)

The expected names should be those constructed in the following chunk

In [10]:
measures = ["year", "month", "activity", "employment", "unemployment", "youth unemployment"]
col_names = ([ "all_"+m for m in measures ] + 
        [ "m_" +m for m in measures[:-1] ] + 
        [ "f_" + m for m in measures[:-1] ])
col_names

['all_year',
 'all_month',
 'all_activity',
 'all_employment',
 'all_unemployment',
 'all_youth unemployment',
 'm_year',
 'm_month',
 'm_activity',
 'm_employment',
 'm_unemployment',
 'f_year',
 'f_month',
 'f_activity',
 'f_employment',
 'f_unemployment']

Check (and fix) the following issues:

- are the column names correct and meaningful? if not, redefine the names

In [19]:
col_dict = {}
i=0
for col in df_file.columns:
    col_dict[col] = col_names[i]
    i+=1
renamed_df = df_file.rename(columns = col_dict)
renamed_df

,all_year,all_month,all_activity,all_employment,all_unemployment,all_youth unemployment,m_year,m_month,m_activity,m_employment,m_unemployment,f_year,f_month,f_activity,f_employment,f_unemployment
0,NaN,NaN,15-64 anni,15-64 anni,Totale,15-24 anni,NaN,NaN,15-64 anni,15-64 anni,Totale,NaN,NaN,15-64 anni,15-64 anni,Totale
1,Maschi e Femmine,NaN,NaN,NaN,NaN,NaN,Maschi,NaN,NaN,NaN,NaN,Femmine,NaN,NaN,NaN,NaN
2,2004,Gennaio,62.51201,57.251936,8.339635,22.883,2004,Gennaio,74.232631,69.486015,6.332135,2004,Gennaio,50.899031,45.130214,11.268679
3,NaN,Febbraio,62.363788,57.312528,8.032435,22.372344,NaN,Febbraio,74.183267,69.50436,6.24335,NaN,Febbraio,50.651898,45.231674,10.651987
4,NaN,Marzo,62.758419,57.471105,8.334358,23.484326,NaN,Marzo,74.525981,69.581998,6.551786,NaN,Marzo,51.097503,45.469969,10.93423
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
262,NaN,Settembre,66.567969,62.507472,5.976135,21.598656,NaN,Settembre,75.365824,70.991556,5.691752,NaN,Settembre,57.6981,53.953941,6.353743
263,NaN,Ottobre,66.504865,62.577162,5.759303,20.317609,NaN,Ottobre,75.283654,71.029682,5.495228,NaN,Ottobre,57.651181,54.05253,6.109835
264,NaN,Novembre,66.233935,62.445141,5.586209,19.393442,NaN,Novembre,75.127971,70.955725,5.400341,NaN,Novembre,57.260434,53.858519,5.834516
265,NaN,Dicembre,66.184713,62.428396,5.517798,20.861006,NaN,Dicembre,75.022661,70.844568,5.404148,NaN,Dicembre,57.264904,53.934269,5.669447


- are there useless (empty or duplicated) columns?
  - if so, remove them

In [25]:
df_unique_col = renamed_df.drop(columns=["m_year","m_month","f_year","f_month"],
                                index= [0,1])
df_unique_col

,all_year,all_month,all_activity,all_employment,all_unemployment,all_youth unemployment,m_activity,m_employment,m_unemployment,f_activity,f_employment,f_unemployment
2,2004,Gennaio,62.51201,57.251936,8.339635,22.883,74.232631,69.486015,6.332135,50.899031,45.130214,11.268679
3,NaN,Febbraio,62.363788,57.312528,8.032435,22.372344,74.183267,69.50436,6.24335,50.651898,45.231674,10.651987
4,NaN,Marzo,62.758419,57.471105,8.334358,23.484326,74.525981,69.581998,6.551786,51.097503,45.469969,10.93423
5,NaN,Aprile,62.322169,57.220141,8.114749,23.843538,74.262768,69.552238,6.27222,50.489089,44.999088,10.825595
6,NaN,Maggio,62.820424,57.648951,8.141401,24.125718,74.802035,69.858516,6.516743,50.945857,45.548466,10.528557
...,...,...,...,...,...,...,...,...,...,...,...,...
262,NaN,Settembre,66.567969,62.507472,5.976135,21.598656,75.365824,70.991556,5.691752,57.6981,53.953941,6.353743
263,NaN,Ottobre,66.504865,62.577162,5.759303,20.317609,75.283654,71.029682,5.495228,57.651181,54.05253,6.109835
264,NaN,Novembre,66.233935,62.445141,5.586209,19.393442,75.127971,70.955725,5.400341,57.260434,53.858519,5.834516
265,NaN,Dicembre,66.184713,62.428396,5.517798,20.861006,75.022661,70.844568,5.404148,57.264904,53.934269,5.669447


- in the ISTAT dataframe, are there incomplete columns to fill?

In [26]:
df_fill = df_unique_col.ffill()
df_fill

,all_year,all_month,all_activity,all_employment,all_unemployment,all_youth unemployment,m_activity,m_employment,m_unemployment,f_activity,f_employment,f_unemployment
2,2004,Gennaio,62.51201,57.251936,8.339635,22.883,74.232631,69.486015,6.332135,50.899031,45.130214,11.268679
3,2004,Febbraio,62.363788,57.312528,8.032435,22.372344,74.183267,69.50436,6.24335,50.651898,45.231674,10.651987
4,2004,Marzo,62.758419,57.471105,8.334358,23.484326,74.525981,69.581998,6.551786,51.097503,45.469969,10.93423
5,2004,Aprile,62.322169,57.220141,8.114749,23.843538,74.262768,69.552238,6.27222,50.489089,44.999088,10.825595
6,2004,Maggio,62.820424,57.648951,8.141401,24.125718,74.802035,69.858516,6.516743,50.945857,45.548466,10.528557
...,...,...,...,...,...,...,...,...,...,...,...,...
262,2025,Settembre,66.567969,62.507472,5.976135,21.598656,75.365824,70.991556,5.691752,57.6981,53.953941,6.353743
263,2025,Ottobre,66.504865,62.577162,5.759303,20.317609,75.283654,71.029682,5.495228,57.651181,54.05253,6.109835
264,2025,Novembre,66.233935,62.445141,5.586209,19.393442,75.127971,70.955725,5.400341,57.260434,53.858519,5.834516
265,2025,Dicembre,66.184713,62.428396,5.517798,20.861006,75.022661,70.844568,5.404148,57.264904,53.934269,5.669447


- are there columns that should have a different type (categorical)?
  - when converting a string to a categorical, be careful:
    - `pd.Categorical()` without specifying categories orders levels alphabetically
    - passing an explicit `categories` list preserves the order of appearance in the dataset

In [ ]:
df_converted = df_fill.copy()
months = ["Gennaio","Febbraio","Marzo","Aprile","Maggio","Giugno","Luglio","Agosto","Settembre","Ottobre","Novembre","Dicembre"]
df_converted["all_month"] = pd.Categorical(df_fill["all_month"],
                                            categories= months,
                                            ordered= True)
df_converted = df_converted.convert_dtypes()

,all_year,all_month,all_activity,all_employment,all_unemployment,all_youth unemployment,m_activity,m_employment,m_unemployment,f_activity,f_employment,f_unemployment
2,2004,Gennaio,62.51201,57.251936,8.339635,22.883,74.232631,69.486015,6.332135,50.899031,45.130214,11.268679
3,2004,Febbraio,62.363788,57.312528,8.032435,22.372344,74.183267,69.50436,6.24335,50.651898,45.231674,10.651987
4,2004,Marzo,62.758419,57.471105,8.334358,23.484326,74.525981,69.581998,6.551786,51.097503,45.469969,10.93423
5,2004,Aprile,62.322169,57.220141,8.114749,23.843538,74.262768,69.552238,6.27222,50.489089,44.999088,10.825595
6,2004,Maggio,62.820424,57.648951,8.141401,24.125718,74.802035,69.858516,6.516743,50.945857,45.548466,10.528557
...,...,...,...,...,...,...,...,...,...,...,...,...
262,2025,Settembre,66.567969,62.507472,5.976135,21.598656,75.365824,70.991556,5.691752,57.6981,53.953941,6.353743
263,2025,Ottobre,66.504865,62.577162,5.759303,20.317609,75.283654,71.029682,5.495228,57.651181,54.05253,6.109835
264,2025,Novembre,66.233935,62.445141,5.586209,19.393442,75.127971,70.955725,5.400341,57.260434,53.858519,5.834516
265,2025,Dicembre,66.184713,62.428396,5.517798,20.861006,75.022661,70.844568,5.404148,57.264904,53.934269,5.669447


## Tidy the data

---

### Note

The `melt()` function is able to unpivot a DataFrame from wide to long format.
Combined with string `str.split()`, it allows extracting multiple names from columns by specifying a separator (which splits the column names) and providing multiple columns where the respective fragments will be placed.

In [36]:
# example
alt_pd = pd.DataFrame({
  "id": [1, 2, 3, 4],
  "height_before": [2, 5, 3, 8],
  "height_after": [3, 5, 6, 9],
  "width_before": [12, 22, 15, 15],
  "width_after": [13, 21, 16, 18]
})
print(alt_pd)
alt_pd_l = alt_pd.melt(id_vars="id", var_name="variable", value_name="value")
alt_pd_l[["dim", "time"]] = alt_pd_l["variable"].str.split("_", expand=True)
alt_pd_l = alt_pd_l.drop(columns="variable")
alt_pd_l

   id  height_before  height_after  width_before  width_after
0   1              2             3            12           13
1   2              5             5            22           21
2   3              3             6            15           16
3   4              8             9            15           18


,id,value,dim,time
0,1,2,height,before
1,2,5,height,before
2,3,3,height,before
3,4,8,height,before
4,1,3,height,after
5,2,5,height,after
6,3,6,height,after
7,4,9,height,after
8,1,12,width,before
9,2,22,width,before


Sometimes this separation can lead to splitting the same observation across multiple rows (for example `dim` is not a true variable), so it is necessary to apply a `pivot_table()` to rearrange the data frame.

In [43]:
# example
alt_pd_l.pivot_table(index=["id", "time"], columns="dim", values="value").reset_index()

dim,id,time,height,width
0,1,after,3.0,13.0
1,1,before,2.0,12.0
2,2,after,5.0,21.0
3,2,before,5.0,22.0
4,3,after,6.0,16.0
5,3,before,3.0,15.0
6,4,after,9.0,18.0
7,4,before,8.0,15.0


----

- In the ISTAT dataset, are there variables spread across multiple columns?
  - once reunited, do they need to be redistributed?

In [ ]:
df_long = df_converted.melt(
    id_vars=["all_year","all_month"],
    var_name="variable",
    value_name="value"
)
df_long
#Splitted step in 2 cell to be more understandable

,all_year,all_month,variable,value
0,2004,Gennaio,all_activity,62.51201
1,2004,Febbraio,all_activity,62.363788
2,2004,Marzo,all_activity,62.758419
3,2004,Aprile,all_activity,62.322169
4,2004,Maggio,all_activity,62.820424
...,...,...,...,...
2645,2025,Settembre,f_unemployment,6.353743
2646,2025,Ottobre,f_unemployment,6.109835
2647,2025,Novembre,f_unemployment,5.834516
2648,2025,Dicembre,f_unemployment,5.669447


In [ ]:
#Separate the variable column in "gender" and "measure" columns still in long format
df_long[["gender","measure"]] = df_long["variable"].str.split("_",expand=True)
df_long = df_long.drop(columns="variable")
df_long

,all_year,all_month,value,gender,measure
0,2004,Gennaio,62.51201,all,activity
1,2004,Febbraio,62.363788,all,activity
2,2004,Marzo,62.758419,all,activity
3,2004,Aprile,62.322169,all,activity
4,2004,Maggio,62.820424,all,activity
...,...,...,...,...,...
2645,2025,Settembre,6.353743,f,unemployment
2646,2025,Ottobre,6.109835,f,unemployment
2647,2025,Novembre,5.834516,f,unemployment
2648,2025,Dicembre,5.669447,f,unemployment


In [69]:
#Rename the columns since now they are generic and the gender is indicated in specific column
df_long = df_long.rename({"all_year":"year",
                "all_month":"month"},
                axis=1)
df_long
#Back to wide format
tidy_df = df_long.pivot_table(
    index= ["year","month","gender"],
    columns="measure",
    values="value"
)
tidy_df = tidy_df.reset_index().rename_axis(None,axis=1)
tidy_df

,year,month,gender,activity,employment,unemployment,youth unemployment
0,2004,Gennaio,all,62.51201,57.251936,8.339635,22.883
1,2004,Gennaio,f,50.899031,45.130214,11.268679,<NA>
2,2004,Gennaio,m,74.232631,69.486015,6.332135,<NA>
3,2004,Febbraio,all,62.363788,57.312528,8.032435,22.372344
4,2004,Febbraio,f,50.651898,45.231674,10.651987,<NA>
...,...,...,...,...,...,...,...
790,2025,Dicembre,f,57.264904,53.934269,5.669447,<NA>
791,2025,Dicembre,m,75.022661,70.844568,5.404148,<NA>
792,2026,Gennaio,all,66.085483,62.603115,5.133663,18.913953
793,2026,Gennaio,f,57.009141,53.874763,5.342522,<NA>


- are there columns resulting from tidying that should have a different type (categorical)?

In [ ]:
#Convert "gender" column type from str to categorical
tidy_df["gender"] = pd.Categorical(
    tidy_df["gender"],
    ["all","m","f"],
    ordered = True
)
tidy_df.dtypes


year                     Int64
month                 category
gender                category
activity               Float64
employment             Float64
unemployment           Float64
youth unemployment     Float64
dtype: object

To represent data in a time series, it is necessary to group year and month into a single variable, so that they are correctly ordered.

One possibility is to transform them into a date, taking the first day of the month as the conventional date.

If the levels of the (categorical) variable *month* are in the correct order (i.e. `"Gennaio", "Febbraio", ... "Dicembre"`), applying `.cat.codes + 1` will yield the numbers `1, 2, ... 12`, since category codes are numbered starting from 0.

In [92]:
df_date = tidy_df.copy()
months_num = df_date["month"].cat.codes+1
df_date["date"] = pd.to_datetime(
    dict(
        year = df_date["year"],
        month = months_num,
        day = 1
    ),
    errors="coerce"
)
df_date = df_date.drop(columns=["year","month"])
df_date = df_date.set_index(["date","gender"]).sort_values(by="date")
df_date

activity  employment  unemployment  youth unemployment
date       gender                                                         
2004-01-01 all      62.51201   57.251936      8.339635              22.883
           f       50.899031   45.130214     11.268679                <NA>
           m       74.232631   69.486015      6.332135                <NA>
2004-02-01 all     62.363788   57.312528      8.032435           22.372344
           f       50.651898   45.231674     10.651987                <NA>
...                      ...         ...           ...                 ...
2025-12-01 all     66.184713   62.428396      5.517798           20.861006
           m       75.022661   70.844568      5.404148                <NA>
2026-01-01 f       57.009141   53.874763      5.342522                <NA>
           all     66.085483   62.603115      5.133663           18.913953
           m       75.079157   71.251968      4.977837                <NA>

[795 rows x 4 columns]